# Preprocessing & Alignment

Reprojects all satellite layers (native resolutions spanning 15m–36km) onto a unified 1km analysis grid using MODIS LST as the master grid, then flattens and merges everything into a single tabular dataset (`Final_dataset.csv`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rasterio --quiet

In [ ]:
# CELL 5: Align rasters and build final dataset

import rasterio
from rasterio.warp import reproject, Resampling
import glob
import numpy as np
import pandas as pd
import os

BASE_PATH = "/content/drive/MyDrive/mini_project_1"

nd_files = sorted(glob.glob(os.path.join(BASE_PATH, "NDVI_NDMI_*.tif")))

all_data = []

for nd_path in nd_files:

    filename = os.path.basename(nd_path)
    parts = filename.replace('.tif', '').split('_')
    year, month = int(parts[2]), int(parts[3])
    key = f"{year}_{month:02d}"

    lst_path = os.path.join(BASE_PATH, f"LST_{key}.tif")
    precip_path = os.path.join(BASE_PATH, f"PRECIP_{key}.tif")
    smap_path = os.path.join(BASE_PATH, f"SMAP_{key}.tif")
    s1_path = os.path.join(BASE_PATH, f"S1_{key}.tif")
    ch4_path = os.path.join(BASE_PATH, f"CH4_{key}.tif")

    if not os.path.exists(lst_path):
        print("Skipping (no LST):", key)
        continue

    #  Read LST as master grid
    with rasterio.open(lst_path) as lst_src:
        lst = lst_src.read(1).astype(np.float32)
        dst_transform = lst_src.transform
        dst_crs = lst_src.crs
        dst_shape = lst.shape

    #  Reprojection function
    def reproject_to_lst(src_path, band=1):
        if not os.path.exists(src_path):
            return np.full(dst_shape, np.nan, dtype=np.float32)

        dst_array = np.full(dst_shape, np.nan, dtype=np.float32)

        with rasterio.open(src_path) as src:
            reproject(
                source=rasterio.band(src, band),
                destination=dst_array,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.average,
                dst_nodata=np.nan
            )
        return dst_array

    #  Reproject all layers
    ndvi = reproject_to_lst(nd_path, 1)
    ndmi = reproject_to_lst(nd_path, 2)
    precip = reproject_to_lst(precip_path)
    smap = reproject_to_lst(smap_path)
    vv = reproject_to_lst(s1_path, 1)
    vh = reproject_to_lst(s1_path, 2)
    ch4 = reproject_to_lst(ch4_path)

    #  Flatten
    ndvi = ndvi.ravel()
    ndmi = ndmi.ravel()
    lst = lst.ravel()
    precip = precip.ravel()
    smap = smap.ravel()
    vv = vv.ravel()
    vh = vh.ravel()
    ch4 = ch4.ravel()

    #  Ratio
    vv_vh_ratio = vv / (vh + 1e-6)

    #  Strong valid mask + physical limits
    valid = (
        (ndvi > -1) & (ndvi < 1) &
        (ndmi > -1) & (ndmi < 1) &
        (lst > -50) & (lst < 70) &
        ~np.isnan(precip) &
        ~np.isnan(smap) &
        ~np.isnan(vv) &
        ~np.isnan(vh) &
        ~np.isnan(ch4)
    )

    if valid.sum() == 0:
        print("No valid pixels:", key)
        continue

    #  Build DataFrame
    chunk_df = pd.DataFrame({
        "year": year,
        "month": month,
        "ndvi": ndvi[valid],
        "lst": lst[valid],
        "precip": precip[valid],
        "smap": smap[valid],
        "vv": vv[valid],
        "vh": vh[valid],
        "vv_vh_ratio": vv_vh_ratio[valid],
        "ch4": ch4[valid],
        "ndmi": ndmi[valid]
    })

    all_data.append(chunk_df)

    print(f"Aligned & processed: {key} | Pixels: {valid.sum()}")

#  Combine all months
df = pd.concat(all_data, ignore_index=True)

# Ensure numeric dtype
df = df.astype({
    "year": "int32",
    "month": "int32",
    "ndvi": "float32",
    "lst": "float32",
    "precip": "float32",
    "smap": "float32",
    "vv": "float32",
    "vh": "float32",
    "vv_vh_ratio": "float32",
    "ch4": "float32",
    "ndmi": "float32"
})

#  Save CSV
csv_path = os.path.join(BASE_PATH, "Final_dataset.csv")
df.to_csv(csv_path, index=False)

print("\nDataset saved at:", csv_path)
print("Total rows:", len(df))
print("CSV size (MB):", round(os.path.getsize(csv_path)/1e6, 2))

No valid pixels: 2017_01
No valid pixels: 2017_02
No valid pixels: 2017_03
No valid pixels: 2017_04
No valid pixels: 2017_05
No valid pixels: 2017_06
No valid pixels: 2017_10
No valid pixels: 2017_11
No valid pixels: 2017_12
No valid pixels: 2018_01
No valid pixels: 2018_02
No valid pixels: 2018_03
No valid pixels: 2018_04
No valid pixels: 2018_05
No valid pixels: 2018_06
No valid pixels: 2018_10
No valid pixels: 2018_11
Aligned & processed: 2018_12 | Pixels: 458
Aligned & processed: 2019_01 | Pixels: 1898
Aligned & processed: 2019_02 | Pixels: 2257
Aligned & processed: 2019_03 | Pixels: 1909
Aligned & processed: 2019_04 | Pixels: 3631
Aligned & processed: 2019_05 | Pixels: 5123
Aligned & processed: 2019_06 | Pixels: 5083
No valid pixels: 2019_07
No valid pixels: 2019_08
No valid pixels: 2019_09
Aligned & processed: 2019_10 | Pixels: 733
Aligned & processed: 2019_11 | Pixels: 2194
Aligned & processed: 2019_12 | Pixels: 1549
Aligned & processed: 2020_01 | Pixels: 2258
Aligned & processe

## Exploratory Data Analysis (EDA)

Quick checks on the final aligned dataset before modeling.

In [ ]:
import pandas as pd

BASE_PATH = "/content/drive/MyDrive/mini_project_1"
csv_path = f"{BASE_PATH}/Final_dataset.csv"

df = pd.read_csv(csv_path)

In [ ]:
print("Total rows:", len(df))
print(df['year'].min(), df['year'].max())
print(df.groupby('year').size())

Total rows: 123417
2018 2022
year
2018      458
2019    24377
2020    32638
2021    40406
2022    25538
dtype: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123417 entries, 0 to 123416
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   year         123417 non-null  int64  
 1   month        123417 non-null  int64  
 2   ndvi         123417 non-null  float64
 3   lst          123417 non-null  float64
 4   precip       123417 non-null  float64
 5   smap         123417 non-null  float64
 6   vv           123417 non-null  float64
 7   vh           123417 non-null  float64
 8   vv_vh_ratio  123417 non-null  float64
 9   ch4          123417 non-null  float64
 10  ndmi         123417 non-null  float64
dtypes: float64(9), int64(2)
memory usage: 10.4 MB


In [ ]:
df.describe()

,year,month,ndvi,lst,precip,smap,vv,vh,vv_vh_ratio,ch4,ndmi
count,123417.000000,123417.000000,123417.000000,123417.000000,123417.000000,123417.000000,123417.000000,123417.000000,123417.000000,123417.000000,123417.000000
mean,2020.536304,5.539302,0.383337,32.020757,65.680941,6.631122,-11.269409,-18.430882,0.611632,1909.953135,0.001144
std,1.039154,3.765702,0.155359,7.450285,95.625145,4.990546,1.302824,1.914278,0.037157,23.689460,0.132203
min,2018.000000,1.000000,0.030526,18.070000,0.000000,1.847273,-17.989859,-28.059113,0.422770,1838.228500,-0.281060
25%,2020.000000,3.000000,0.255699,25.100000,4.715836,2.733510,-12.103478,-19.570436,0.585849,1890.086900,-0.108948
50%,2021.000000,4.000000,0.348486,29.910000,27.578005,4.487300,-11.334920,-18.346334,0.611175,1910.184100,-0.028362
75%,2021.000000,10.000000,0.503873,39.610000,96.919520,9.058592,-10.477659,-17.213655,0.638429,1925.820000,0.097336
max,2022.000000,12.000000,0.843587,49.090000,784.970300,24.860933,-5.034234,-11.543833,0.754885,1977.785800,0.439209


In [ ]:
df.corr()

,year,month,ndvi,lst,precip,smap,vv,vh,vv_vh_ratio,ch4,ndmi
year,1.000000,-0.233523,0.195390,-0.121004,0.034792,0.037808,0.101836,0.140713,0.036754,0.478882,0.142385
month,-0.233523,1.000000,-0.022144,-0.170256,0.017134,0.383814,0.239564,0.165860,-0.173145,0.555494,0.103706
ndvi,0.195390,-0.022144,1.000000,-0.487152,0.052319,0.559348,0.318766,0.618540,0.435344,0.206294,0.905211
lst,-0.121004,-0.170256,-0.487152,1.000000,0.089993,-0.430483,-0.387850,-0.435586,0.014854,-0.299010,-0.589572
precip,0.034792,0.017134,0.052319,0.089993,1.000000,0.453477,0.108776,0.047730,-0.124473,0.007401,0.045794
smap,0.037808,0.383814,0.559348,-0.430483,0.453477,1.000000,0.403636,0.455031,-0.003001,0.483117,0.601752
vv,0.101836,0.239564,0.318766,-0.387850,0.108776,0.403636,1.000000,0.856065,-0.471443,0.243001,0.279449
vh,0.140713,0.165860,0.618540,-0.435586,0.047730,0.455031,0.856065,1.000000,0.049312,0.226215,0.515380
vv_vh_ratio,0.036754,-0.173145,0.435344,0.014854,-0.124473,-0.003001,-0.471443,0.049312,1.000000,-0.079113,0.337810
ch4,0.478882,0.555494,0.206294,-0.299010,0.007401,0.483117,0.243001,0.226215,-0.079113,1.000000,0.315218
